# TP Mentoré : Exploration & Profiling Approfondi d'un Dataset Réel (4h)
## Cas d'usage : Open Food Facts France (100 000 produits)

### Contexte métier
Vous intégrez l'équipe Data Platform. Votre premier rôle n'est pas de nettoyer à l'aveugle, mais de livrer un **audit de santé complet (Data Quality Assessment)** d'une source brute avant son intégration dans l'entrepôt de données.

---
### Découpage des 6 missions :
1. **Mission 1 — Ingestion robuste & Optimisation mémoire (Downcasting)** (40 min)
2. **Mission 2 — Matrice de complétude & Collisions d'identifiants** (30 min)
3. **Mission 3 — Fouille textuelle & Tags de catégories** (35 min)
4. **Mission 4 — Chasse aux anomalies physiques & Outliers statistiques** (45 min)
5. **Mission 5 — Industrialisation de l'audit (`generer_rapport_qualite`)** (40 min)
6. **Mission 6 — Extraction prioritaire, Multi-indexation & Persistance Parquet** (30 min)

In [ ]:
import pandas as pd
import numpy as np
import os

print("Environnement d'audit prêt !")

## Mission 1 : Ingestion robuste & Optimisation mémoire (40 min)

Un Data Engineer ne laisse jamais Pandas inférer les types au hasard sur des volumes importants.

### Consignes :
1. Chargez `openfoodfacts_france_sample.csv` (séparateur `;`) en forçant la colonne `code` en chaîne de caractères (`str`).
2. Mesurez la mémoire brute totale en Mo dans `memoire_initiale_mo` (via `memory_usage(deep=True)`).
3. **Optimisation :**
   - Convertissez la colonne `nutriscore_grade` en type `category`.
   - Convertissez les colonnes nutritionnelles numériques (`fat_100g`, `sugars_100g`, `proteins_100g`, `salt_100g`) de `float64` vers `float32` via `.astype('float32')`.
4. Mesurez la nouvelle mémoire dans `memoire_optimisee_mo` et calculez le pourcentage de mémoire économisé.

In [ ]:
# TODO : Chargement et mesures
df_raw = ...
memoire_initiale_mo = ...

# Optimisations de typage   
# ...

memoire_optimisee_mo = ...
gain_pct = ((memoire_initiale_mo - memoire_optimisee_mo) / memoire_initiale_mo) * 100

print(f"RAM initiale   : {memoire_initiale_mo:.2f} Mo")
print(f"RAM optimisée  : {memoire_optimisee_mo:.2f} Mo")
print(f"Gain de mémoire : {gain_pct:.1f} %")

In [ ]:
assert df_raw.shape[0] == 100_000, "Le dataset doit faire 100 000 lignes."
assert df_raw["code"].dtype in ["object", "string"], "Le code-barres doit être typé en texte."
assert str(df_raw["nutriscore_grade"].dtype) == "category", "nutriscore_grade doit être de type category."
assert df_raw["sugars_100g"].dtype == "float32", "sugars_100g doit être encodé en float32."
assert memoire_optimisee_mo < memoire_initiale_mo, "L'optimisation doit réduire l'empreinte mémoire."
print("✅ Mission 1 validée !")

## Mission 2 : Matrice de complétude & Collisions d'identifiants (30 min)

### Consignes :
1. Calculez le taux de valeurs manquantes (en %) pour chaque colonne et stockez-le dans `taux_nan` (Series triée par ordre décroissant).
2. Déterminez la liste des colonnes ayant **plus de 40% de données manquantes** dans `colonnes_critiques_vides`.
3. Comptez les doublons stricts (`nb_doublons_stricts`).
4. Isolez les produits dont le code-barres est dupliqué (`subset=['code']`) : enregistrez ces lignes dupliquées dans un DataFrame `df_collisions_code`.

In [ ]:
# TODO : Complétude et doublons
taux_nan = ...
colonnes_critiques_vides = ...

nb_doublons_stricts = ...
df_collisions_code = ...

print("Colonnes critiques (>40% NaN) :", colonnes_critiques_vides)
print(f"Nombre de lignes impliquées dans une collision de code-barres : {len(df_collisions_code)}")

In [ ]:
assert isinstance(taux_nan, pd.Series), "taux_nan doit être une Series."
assert isinstance(colonnes_critiques_vides, list), "colonnes_critiques_vides doit être une liste de noms de colonnes."
assert isinstance(df_collisions_code, pd.DataFrame), "df_collisions_code doit être un DataFrame."
assert df_collisions_code["code"].duplicated().any() or len(df_collisions_code) == 0, "Vérifiez la détection des collisions de code-barres."
print("✅ Mission 2 validée !")

## Mission 3 : Fouille textuelle & Tags de catégories (35 min)

La colonne `categories_tags` contient des tags normalisés séparés par des virgules (ex: `en:beverages,en:sodas`).

### Consignes :
1. Créez un masque booléen identifiant les produits appartenant à la catégorie des boissons (contenant la sous-chaîne `'beverages'`).
2. Décomptez le nombre total de boissons identifiées dans `nb_boissons`.
3. Pour les produits qui ont une catégorie non nulle, calculez le nombre moyen de catégories associées par produit dans `nb_moyen_categories` (Indice : utilisez `.str.split(',').str.len()`).

In [ ]:
# TODO : Exploration textuelle
masque_boissons = ...
nb_boissons = ...

nb_moyen_categories = ...

print(f"Nombre de boissons répertoriées : {nb_boissons}")
print(f"Nombre moyen de catégories par produit renseigné : {nb_moyen_categories:.2f}")

In [ ]:
assert isinstance(nb_boissons, (int, np.integer)), "nb_boissons doit être un entier."
assert nb_boissons > 0, "Aucune boisson n'a été détectée. Vérifiez le motif recherché."
assert isinstance(nb_moyen_categories, (float, np.floating)), "nb_moyen_categories doit être un float."
assert 1.0 <= nb_moyen_categories <= 15.0, "Le nombre moyen de tags par produit semble incohérent."
print("✅ Mission 3 validée !")

## Mission 4 : Anomalies physiques & Détection d'outliers (45 min)

### Consignes :
1. **Incohérence de masse :** Isolez dans `df_aberrants_masse` les lignes où la somme `fat_100g + sugars_100g + proteins_100g + salt_100g` dépasse **105 g**.
2. **Valeurs négatives :** Comptez combien de produits ont au moins un nutriment strictement négatif dans `nb_nutriments_negatifs`.
3. **Outliers statistiques sur le sucre :**
   - Calculez le 99ᵉ percentile (`quantile(0.99)`) de la colonne `sugars_100g` dans `seuil_sucre_q99`.
   - Isolez dans `df_sucre_extreme` les produits dont la teneur en sucre dépasse ce quantile tout en restant physiquement possible (<= 100 g).


In [ ]:
# TODO : Contrôles d'intégrité
somme_nutri = ...
df_aberrants_masse = ...

nb_nutriments_negatifs = ...

seuil_sucre_q99 = ...
df_sucre_extreme = ...

print(f"Produits violant la limite de masse : {len(df_aberrants_masse)}")
print(f"Produits avec au moins un nutriment < 0 : {nb_nutriments_negatifs}")
print(f"Seuil sucre (Q99) : {seuil_sucre_q99:.2f} g/100g")
print(f"Nombre de produits à teneur extrême en sucre : {len(df_sucre_extreme)}")

In [ ]:
assert isinstance(df_aberrants_masse, pd.DataFrame), "df_aberrants_masse doit être un DataFrame."
assert isinstance(nb_nutriments_negatifs, (int, np.integer)), "nb_nutriments_negatifs doit être un entier."
assert 0 <= seuil_sucre_q99 <= 100, "Le quantile 99% du sucre doit être compris entre 0 et 100g."
assert (df_sucre_extreme["sugars_100g"] > seuil_sucre_q99).all(), "Tous les produits filtrés doivent dépasser le seuil Q99."
print("✅ Mission 4 validée !")

## Mission 5 : Industrialisation de l'audit (`generer_rapport_qualite`) (40 min)

En entreprise, on ne fait pas d'audits manuels à chaque nouveau fichier. On code des outils d'inspection réutilisables.

### Consignes :
Développez la fonction `generer_rapport_qualite(df: pd.DataFrame) -> pd.DataFrame` qui prend en entrée n'importe quel DataFrame et renvoie un tableau récapitulatif indexé par le nom des colonnes avec les informations suivantes :
- `type_donnees` : le type (`str(df[col].dtype)`)
- `taux_manquants_pct` : pourcentage de NaN arrondi à 2 décimales
- `nb_valeurs_uniques` : nombre de modalités distinctes (`nunique()`)
- `memoire_ko` : taille occupée en mémoire par la colonne en Ko (`memory_usage(deep=True)` / 1024 arrondi à 1 décimale)

In [ ]:
# TODO : Coder la fonction de diagnostic générique
def generer_rapport_qualite(df: pd.DataFrame) -> pd.DataFrame:
    rapport = {}
    # Bouclez sur les colonnes du DataFrame et construisez le dictionnaire de métriques
    # ...
    return pd.DataFrame.from_dict(rapport, orient="index")

df_audit = generer_rapport_qualite(df_raw)
df_audit

In [ ]:
assert isinstance(df_audit, pd.DataFrame), "La fonction doit renvoyer un DataFrame."
assert len(df_audit) == df_raw.shape[1], "Le rapport doit comporter une ligne par colonne analysée."
assert set(df_audit.columns) == {"type_donnees", "taux_manquants_pct", "nb_valeurs_uniques", "memoire_ko"}, "Les colonnes du rapport ne correspondent pas au cahier des charges."
print("✅ Mission 5 validée : Votre outil d'audit automatique fonctionne !")

## Mission 6 : Extraction prioritaire & Persistance Parquet (30 min)

### Consignes :
1. Construisez le dataset propre `df_gold` selon ces conditions strictes :
   - Code-barres `code` non vide et sans doublon.
   - `product_name` non vide.
   - `nutriscore_grade` valide (`isin(['a', 'b', 'c', 'd', 'e'])`).
   - Énergie positive ou nulle (`energy-kcal_100g >= 0`).
   - Somme des macronutriments $\le 105$ g.
2. Réindexez proprement avec `reset_index(drop=True)`.
3. Sauvegardez le résultat sous `openfoodfacts_gold.parquet` (compression `snappy`).
4. Calculez le ratio de compression entre le fichier CSV source et ce Parquet assaini.

In [ ]:
# TODO : Filtre final combiné et export
filtre_final = ...
df_gold = ...

# Export Parquet
df_gold.to_parquet("openfoodfacts_gold.parquet", compression="snappy")

# Comparaison des tailles disque
taille_csv = os.path.getsize("openfoodfacts_france_sample.csv") / (1024 * 1024)
taille_parquet = os.path.getsize("openfoodfacts_gold.parquet") / (1024 * 1024)
print(f"Dataset Gold : {len(df_gold)} lignes assainies sur 100 000 initiales.")
print(f"Poids CSV initial : {taille_csv:.2f} Mo | Poids Parquet final : {taille_parquet:.2f} Mo (Ratio : x{taille_csv/taille_parquet:.1f})")

In [ ]:
assert os.path.exists("openfoodfacts_gold.parquet"), "Le fichier Parquet n'a pas été créé."
assert len(df_gold) > 10_000, "Le dataset filtré semble anormalement petit."
assert df_gold["code"].duplicated().sum() == 0, "Il subsiste des doublons d'identifiants."
assert (df_gold["energy-kcal_100g"] >= 0).all(), "Il subsiste des calories négatives."
print("🎉 FÉLICITATIONS ! Vous avez brillamment complété l'audit complet du dataset réel !")